In [3]:
import pandas as pd
import numpy as np
from astroquery.mast import Catalogs
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

# 1. Download Catalog Tables

koi_table = NasaExoplanetArchive.query_criteria(
    table="cumulative",      
    select="*"
)
koi_df = koi_table.to_pandas()

toi_table = NasaExoplanetArchive.query_criteria(
    table="toi",             
    select="*"
)
toi_df = toi_table.to_pandas()

tce_df = pd.read_csv("../data/kepler_tce.csv", on_bad_lines='skip')
print(tce_df.shape)

In [18]:
def load_exoplanet_catalogs(tce_csv_path="../data/kepler_tce.csv"):
    """
    Load KOI, TOI, and TCE catalogs. 
    KOI & TOI are fetched via astroquery, while TCE is loaded from a local CSV.
    
    Parameters
    ----------
    tce_csv_path : str
        Path to locally downloaded Kepler TCE CSV file
        (from https://exoplanetarchive.ipac.caltech.edu/cgi-bin/TblView/nph-tblView?app=ExoTbls&config=tce)

    Returns
    -------
    koi_df, toi_df, tce_df, all_df : pd.DataFrame
        Individual KOI, TOI, TCE tables and merged dataset with labels
    """
    
    # KOI catalog
    try:
        print("Fetching KOI cumulative table (Kepler OI)...")
        koi_table = NasaExoplanetArchive.query_criteria(table="cumulative", select="*")
        koi_df = koi_table.to_pandas()

        # Label (confirmed = 1)
        koi_df["label"] = koi_df["koi_disposition"].apply(
            lambda x: 1 if str(x).upper() == "CONFIRMED" else 0
        )
        koi_df["catalog"] = "KOI"

        koi_df = koi_df.rename(
            columns={"kepid": "target_id", "koi_period": "period", "koi_time0bk": "epoch"}
        )
    except Exception as e:
        print(f"KOI fetch failed: {e}")
        koi_df = pd.DataFrame()

    # TOI catalog
    try:
        toi_table = NasaExoplanetArchive.query_criteria(table="toi", select="*")
        toi_df = toi_table.to_pandas()

        # Label confirmed planets
        if "tfopwg_disp" in toi_df.columns:
            toi_df["label"] = toi_df["tfopwg_disp"].apply(
                lambda x: 1 if str(x).upper() in ["CP","KP"] else 0
            )
        else:
            toi_df["label"] = 0

        toi_df["catalog"] = "TOI"

        # Flexible renaming
        rename_map = {}
        if "tid" in toi_df.columns:          # sometimes present
            rename_map["tid"] = "target_id"
        elif "toi" in toi_df.columns:        # fallback
            rename_map["toi"] = "target_id"

        # Use pl_orbper for orbital period
        if "pl_orbper" in toi_df.columns:
            rename_map["pl_orbper"] = "period"
        elif "toi_period" in toi_df.columns:   # alternative schema
            rename_map["toi_period"] = "period"

        # Use pl_tranmid for epoch
        if "pl_tranmid" in toi_df.columns:
            rename_map["pl_tranmid"] = "epoch"
        elif "toi_tranmid" in toi_df.columns:
            rename_map["toi_tranmid"] = "epoch"
        elif "transit_epoch" in toi_df.columns:
            rename_map["transit_epoch"] = "epoch"

        toi_df = toi_df.rename(columns=rename_map)

    except Exception as e:
        print(f"TOI fetch failed: {e}")
        toi_df = pd.DataFrame()

    # TCE catalog (from local CSV)
    try:
        print(f"Loading TCE table from {tce_csv_path} ...")
        tce_df = pd.read_csv(tce_csv_path, on_bad_lines='skip')

        tce_df["label"] = 0  
        tce_df["catalog"] = "TCE"

        tce_df = tce_df.rename(
            columns={"kepid": "target_id", "tce_period": "period", "tce_time0bk": "epoch"}
        )
    except Exception as e:
        print(f"TCE load failed: {e}")
        tce_df = pd.DataFrame()

    # Merge results
    all_df = pd.concat([koi_df, toi_df, tce_df], ignore_index=True, sort=False)
    print("Final merged dataset shape:", all_df.shape)

    return koi_df, toi_df, tce_df, all_df

In [19]:
koi_df, toi_df, tce_df, all_df = load_exoplanet_catalogs("../data/kepler_tce.csv")

Fetching KOI cumulative table (Kepler OI)...
Loading TCE table from ../data/kepler_tce.csv ...
Final merged dataset shape: (51264, 266)


In [20]:
koi_df.to_csv("../data/koi_df.csv")
toi_df.to_csv("../data/toi_df.csv")
tce_df.to_csv("../data/tce_df.csv")
all_df.to_csv("../data/all_df.csv")

In [21]:
print(toi_df[["target_id","period","epoch","label"]].head())
print(toi_df["label"].value_counts())

   target_id     period         epoch  label
0   79748331   6.443866  2.458657e+06      1
1   79748331  12.226560  2.458664e+06      1
2    7088246   2.160344  2.458654e+06      1
3  201604954   4.606184  2.458658e+06      1
4  201642601   3.131674  2.458657e+06      1
label
0    6436
1    1232
Name: count, dtype: int64


BaseLine Model

In [5]:
def get_label(row):
    if row["koi_disposition"] == "CONFIRMED":
        return 1
    if row["koi_disposition"] == "FALSE POSITIVE":
        return 0
    if str(row["toidisplay"]).lower() == "planet candidate":
        return 1
    if str(row["toidisplay"]).lower() == "false positive":
        return 0
    if str(row["tfopwg_disp"]).upper() == "PC":
        return 1
    if str(row["tfopwg_disp"]).upper() == "FP":
        return 0
    return None

In [8]:
all_df['label'] = all_df.apply(get_label,axis=1)

In [9]:
train_df = all_df.dropna(subset=["label"])
train_df["label"] = train_df["label"].astype(int)

print(train_df["label"].value_counts())

label
1    7428
0    6027
Name: count, dtype: int64


C:\Users\yashs\AppData\Local\Temp\ipykernel_9408\1857470499.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["label"] = train_df["label"].astype(int)


In [10]:
transit_feats = [
    "koi_period","koi_time0bk","koi_duration","koi_depth","koi_prad","koi_ror",
    "tce_period","tce_time0bk","tce_duration","tce_depth","tce_prad","tce_mes"
]

star_feats = [
    "koi_steff","koi_srad","koi_smass","koi_slogg",
    "tce_steff","tce_sradius","tce_slogg"
]

vetting_feats = [
    "koi_fpflag_nt","koi_fpflag_ss","koi_fpflag_co","koi_fpflag_ec",
    "koi_model_snr","koi_max_mult_ev"
]

feature_cols = transit_feats + star_feats + vetting_feats

In [11]:
available_features = [c for c in feature_cols if c in train_df.columns]

In [12]:
X = train_df[available_features].copy()
y = train_df["label"].copy()

In [13]:
print("Selected features:", available_features)
print("Feature matrix shape:", X.shape)

Selected features: ['koi_duration', 'koi_depth', 'koi_prad', 'koi_ror', 'tce_duration', 'tce_depth', 'tce_prad', 'koi_steff', 'koi_srad', 'koi_smass', 'koi_slogg', 'tce_steff', 'tce_sradius', 'tce_slogg', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec', 'koi_model_snr', 'koi_max_mult_ev']
Feature matrix shape: (13455, 20)


In [43]:
X.to_csv("../data/X.csv")
y.to_csv("../data/y.csv")

In [24]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
from lightgbm import LGBMClassifier
from lightgbm import early_stopping, log_evaluation

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [32]:
clf = LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
    learning_rate=0.05,
    num_leaves=63,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    n_estimators=1000,
    random_state=42
)

clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="auc",
    callbacks=[early_stopping(50), log_evaluation(50)]
)

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Info] Number of positive: 5942, number of negative: 4822
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000506 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2563
[LightGBM] [Info] Number of data poin

,boosting_type,'gbdt'
,num_leaves,63
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,'binary'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [33]:
y_pred_prob = clf.predict_proba(X_test)[:, 1]
y_pred = (y_pred_prob >= 0.5).astype(int)

# Metrics
print("PR-AUC:", average_precision_score(y_test, y_pred_prob))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))
print(classification_report(y_test, y_pred))

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
PR-AUC: 0.9066051596923883
ROC-AUC: 0.932407588390678
              precision    recall  f1-score   support

           0       1.00      0.79      0.88      1205
           1       0.85      1.00      0.92      1486

    accuracy                           0.90      2691
   macro avg       0.92      0.89      0.90      2691
weighted avg       0.92      0.90      0.90      2691

